# Steam Reviews Ingestion Pipeline (Manual Download Version)

This notebook assumes the datasets were **manually downloaded from Kaggle** and placed in the `data/raw/` directory.

## Expected file placement

Place the manually downloaded files in this structure:

```text
data/
└── raw/
    ├── all_reviews.csv
    └── games.csv    # or steam_games.parquet
```

If your filenames differ, just update the paths in **Block 1**.

## Output files

- `data/intermediate/steam_reviews_raw.parquet`
- `data/intermediate/steam_reviews_clean_v2.parquet`
- `data/final/steam_reviews_full_with_genres.parquet`

## Block 1 — Setup and locate the manually downloaded files

In [1]:
%pip install -q pyarrow pandas numpy

from pathlib import Path
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
FINAL_DIR = DATA_DIR / "final"

for d in [DATA_DIR, RAW_DIR, INTERMEDIATE_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Update these if your downloaded filenames are different
reviews_csv_path = RAW_DIR / "all_reviews.csv"
games_path = RAW_DIR / "games.csv"   # can also be a .parquet file

RAW_PARQUET_PATH = INTERMEDIATE_DIR / "all_reviews_raw.parquet"
TYPED_PARQUET_PATH = INTERMEDIATE_DIR / "all_reviews_clean_v2.parquet"
MERGED_PARQUET_PATH = FINAL_DIR / "all_reviews_full_with_genres.parquet"

if not reviews_csv_path.exists():
    raise FileNotFoundError(
        f"Reviews file not found: {reviews_csv_path}\n"
        "Download the Steam Reviews dataset from Kaggle and place the CSV in data/raw/."
    )

if not games_path.exists():
    alt_games_path = RAW_DIR / "games.parquet"
    if alt_games_path.exists():
        games_path = alt_games_path
    else:
        raise FileNotFoundError(
            f"Games metadata file not found: {games_path}\n"
            "Download the Steam Games dataset from Kaggle and place the CSV or Parquet in data/raw/."
        )

print("reviews_csv_path   =", reviews_csv_path)
print("games_path         =", games_path)
print("RAW_PARQUET_PATH   =", RAW_PARQUET_PATH)
print("TYPED_PARQUET_PATH =", TYPED_PARQUET_PATH)
print("MERGED_PARQUET_PATH=", MERGED_PARQUET_PATH)

Note: you may need to restart the kernel to use updated packages.
reviews_csv_path   = d:\DSC 288R Testing\data\raw\all_reviews.csv
games_path         = d:\DSC 288R Testing\data\raw\games.csv
RAW_PARQUET_PATH   = d:\DSC 288R Testing\data\intermediate\all_reviews_raw.parquet
TYPED_PARQUET_PATH = d:\DSC 288R Testing\data\intermediate\all_reviews_clean_v2.parquet
MERGED_PARQUET_PATH= d:\DSC 288R Testing\data\final\all_reviews_full_with_genres.parquet


## Block 2 — Convert the raw reviews CSV into a raw Parquet mirror and validate it

In [2]:
def convert_csv_to_raw_parquet(
    csv_path: str,
    parquet_path: str,
    block_size_bytes: int = 256 * 1024 * 1024,
    compression: str = "zstd",
):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    with open(csv_path, "rb") as f:
        header_reader = pacsv.open_csv(
            f,
            read_options=pacsv.ReadOptions(block_size=64 * 1024),
            convert_options=pacsv.ConvertOptions(strings_can_be_null=True),
        )
        first_batch = header_reader.read_next_batch()
        col_names = first_batch.schema.names

    # Preserve the raw file as faithfully as possible by keeping everything as strings first
    raw_column_types = {col: pa.string() for col in col_names}

    writer = None
    total_rows = 0
    row_groups_written = 0

    with open(csv_path, "rb") as f:
        reader = pacsv.open_csv(
            f,
            read_options=pacsv.ReadOptions(use_threads=True, block_size=block_size_bytes),
            parse_options=pacsv.ParseOptions(delimiter=","),
            convert_options=pacsv.ConvertOptions(
                column_types=raw_column_types,
                strings_can_be_null=True,
            ),
        )

        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break

            if batch is None or batch.num_rows == 0:
                break

            table = pa.Table.from_batches([batch])

            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_path,
                    table.schema,
                    compression=compression,
                    use_dictionary=True,
                )

            writer.write_table(table)
            total_rows += table.num_rows
            row_groups_written += 1

            if row_groups_written % 10 == 0:
                print(f"Wrote {row_groups_written} row groups | rows so far: {total_rows:,}")

    if writer is not None:
        writer.close()

    pf = pq.ParquetFile(parquet_path)
    print("\nRaw parquet complete.")
    print("Rows:", f"{pf.metadata.num_rows:,}")
    print("Row groups:", pf.num_row_groups)
    print(pf.schema)

def sample_check_csv_vs_parquet(csv_path, parquet_path, total_rows, cols=None, seed=232, n_checks=3, sample_rows=2000):
    rng = np.random.default_rng(seed)

    if cols is None:
        cols = pd.read_csv(csv_path, nrows=0).columns.tolist()

    use_cols = cols[:min(10, len(cols))]
    pf = pq.ParquetFile(parquet_path)

    for i in range(n_checks):
        start = int(rng.integers(0, max(1, total_rows - sample_rows)))
        need_start = start
        need_end = start + sample_rows

        csv_slice = pd.read_csv(
            csv_path,
            skiprows=range(1, start + 1),
            nrows=sample_rows,
            usecols=use_cols,
            dtype="string",
        ).reset_index(drop=True)

        taken = []
        seen = 0
        first_rg_start = None

        for rg in range(pf.num_row_groups):
            rg_rows = pf.metadata.row_group(rg).num_rows
            rg_start = seen
            rg_end = seen + rg_rows

            if rg_end > need_start and rg_start < need_end:
                if first_rg_start is None:
                    first_rg_start = rg_start
                taken.append(pf.read_row_group(rg, columns=use_cols).to_pandas().astype("string"))

            seen += rg_rows
            if seen >= need_end:
                break

        pq_block = pd.concat(taken, ignore_index=True)
        offset = need_start - first_rg_start
        pq_slice = pq_block.iloc[offset: offset + sample_rows].reset_index(drop=True)

        csv_norm = csv_slice.astype("string")
        pq_norm = pq_slice.astype("string")

        if not csv_norm.equals(pq_norm):
            diff = (csv_norm != pq_norm) & ~(csv_norm.isna() & pq_norm.isna())
            locs = diff.stack()
            locs = locs[locs].index.tolist()
            raise AssertionError(
                f"Sample check failed on check {i+1}/{n_checks} at start row {start}. "
                f"First diff: {locs[0] if locs else 'unknown'}"
            )

        print(f"[samplecheck] check {i+1}/{n_checks} passed")

    print("[samplecheck] all sample checks passed")

convert_csv_to_raw_parquet(reviews_csv_path, RAW_PARQUET_PATH)

raw_total_rows = pq.ParquetFile(RAW_PARQUET_PATH).metadata.num_rows
#sample_check_csv_vs_parquet(
#    csv_path=reviews_csv_path,
#    parquet_path=RAW_PARQUET_PATH,
#    total_rows=raw_total_rows,
)

Wrote 10 row groups | rows so far: 7,269,899
Wrote 20 row groups | rows so far: 14,574,448
Wrote 30 row groups | rows so far: 21,451,147
Wrote 40 row groups | rows so far: 28,378,247
Wrote 50 row groups | rows so far: 34,967,771
Wrote 60 row groups | rows so far: 42,051,567
Wrote 70 row groups | rows so far: 49,604,428
Wrote 80 row groups | rows so far: 56,323,994
Wrote 90 row groups | rows so far: 63,096,981
Wrote 100 row groups | rows so far: 70,030,461
Wrote 110 row groups | rows so far: 77,584,448
Wrote 120 row groups | rows so far: 84,441,121
Wrote 130 row groups | rows so far: 91,906,991
Wrote 140 row groups | rows so far: 101,203,797
Wrote 150 row groups | rows so far: 108,391,496

Raw parquet complete.
Rows: 113,883,717
Row groups: 165
required group field_id=-1 schema {
  optional binary field_id=-1 recommendationid (String);
  optional binary field_id=-1 appid (String);
  optional binary field_id=-1 game (String);
  optional binary field_id=-1 author_steamid (String);
  optio

## Block 3 — Apply cleaned typing to the raw reviews Parquet

In [3]:
DTYPE_MAP = {
    "recommendationid": "Int64",
    "appid": "Int64",
    "author_steamid": "string",
    "game": "string",
    "language": "string",
    "review": "string",
    "timestamp_created": "datetime64[ns]",
    "timestamp_updated": "datetime64[ns]",
    "voted_up": "boolean",
    "votes_up": "Int64",
    "votes_funny": "Int64",
    "weighted_vote_score": "float64",
    "comment_count": "Int64",
    "steam_purchase": "boolean",
    "received_for_free": "boolean",
    "written_during_early_access": "boolean",
    "hidden_in_steam_china": "boolean",
    "steam_china_location": "string",
    "author_num_games_owned": "Int64",
    "author_num_reviews": "Int64",
    "author_playtime_forever": "Int64",
    "author_playtime_last_two_weeks": "Int64",
    "author_playtime_at_review": "Int64",
    "author_last_played": "datetime64[ns]",
}

def cast_clean_types(df: pd.DataFrame, dtype_map: dict) -> pd.DataFrame:
    df = df.copy()

    datetime_cols = {"timestamp_created", "timestamp_updated", "author_last_played"}
    bool_cols = {
        "voted_up",
        "steam_purchase",
        "received_for_free",
        "written_during_early_access",
        "hidden_in_steam_china",
    }

    for col, target in dtype_map.items():
        if col not in df.columns:
            continue

        if col in datetime_cols:
            df[col] = pd.to_datetime(pd.to_numeric(df[col], errors="coerce"), unit="s", errors="coerce")
        elif col in bool_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8").astype("boolean")
        elif target == "Int64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif target in ("float32", "float64"):
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(target)
        elif target == "string":
            df[col] = df[col].astype("string")
        else:
            df[col] = df[col].astype(target)

    return df

pf = pq.ParquetFile(RAW_PARQUET_PATH)
writer = None
total_rows = 0

for rg in range(pf.num_row_groups):
    df = pf.read_row_group(rg).to_pandas(types_mapper=pd.ArrowDtype)
    df = cast_clean_types(df, DTYPE_MAP)
    table = pa.Table.from_pandas(df, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(
            TYPED_PARQUET_PATH,
            table.schema,
            compression="zstd",
            use_dictionary=True,
        )

    writer.write_table(table)
    total_rows += len(df)

    if (rg + 1) % 10 == 0 or rg == pf.num_row_groups - 1:
        print(f"Processed row group {rg + 1}/{pf.num_row_groups} | rows so far: {total_rows:,}")

if writer is not None:
    writer.close()

typed_pf = pq.ParquetFile(TYPED_PARQUET_PATH)
print("\nTyped parquet complete.")
print("Rows:", f"{typed_pf.metadata.num_rows:,}")
print("Row groups:", typed_pf.num_row_groups)
print(typed_pf.schema)

Processed row group 10/165 | rows so far: 7,269,899
Processed row group 20/165 | rows so far: 14,574,448
Processed row group 30/165 | rows so far: 21,451,147
Processed row group 40/165 | rows so far: 28,378,247
Processed row group 50/165 | rows so far: 34,967,771
Processed row group 60/165 | rows so far: 42,051,567
Processed row group 70/165 | rows so far: 48,770,700
Processed row group 80/165 | rows so far: 55,720,407
Processed row group 90/165 | rows so far: 62,499,854
Processed row group 100/165 | rows so far: 69,474,615
Processed row group 110/165 | rows so far: 76,946,194
Processed row group 120/165 | rows so far: 83,850,587
Processed row group 130/165 | rows so far: 90,483,411
Processed row group 140/165 | rows so far: 97,306,730
Processed row group 150/165 | rows so far: 104,565,207
Processed row group 160/165 | rows so far: 110,818,509
Processed row group 165/165 | rows so far: 113,883,717

Typed parquet complete.
Rows: 113,883,717
Row groups: 165
required group field_id=-1 sch

## Block 4 — Merge typed reviews with metadata and save the final Parquet

In [20]:
reviews_pf = pq.ParquetFile(TYPED_PARQUET_PATH)
writer = None
total_rows = 0
total_missing_genres = 0

def clean_genre_value(x):
    if x is None:
        return None
    if x is pd.NA:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    return x

for rg in range(reviews_pf.num_row_groups):
    df = reviews_pf.read_row_group(rg).to_pandas(types_mapper=pd.ArrowDtype)

    app_ids = pd.to_numeric(df["appid"], errors="coerce").astype("Int64")
    df["Game_Genres"] = app_ids.map(appid_to_genres)

    total_missing_genres += df["Game_Genres"].isna().sum()
    total_rows += len(df)

    normal_cols = [c for c in df.columns if c != "Game_Genres"]
    normal_table = pa.Table.from_pandas(df[normal_cols], preserve_index=False)

    genre_values = [clean_genre_value(x) for x in df["Game_Genres"]]
    genre_array = pa.array(genre_values, type=pa.list_(pa.string()))

    table = normal_table.append_column("Game_Genres", genre_array)

    if writer is None:
        writer = pq.ParquetWriter(
            MERGED_PARQUET_PATH,
            table.schema,
            compression="zstd",
            use_dictionary=True,
        )

    writer.write_table(table)

    if (rg + 1) % 10 == 0 or rg == reviews_pf.num_row_groups - 1:
        print(f"Merged row group {rg + 1}/{reviews_pf.num_row_groups} | rows so far: {total_rows:,}")

if writer is not None:
    writer.close()

merged_pf = pq.ParquetFile(MERGED_PARQUET_PATH)
missing_rate = total_missing_genres / total_rows if total_rows else np.nan

print("\nMerged parquet complete.")
print("Rows:", f"{merged_pf.metadata.num_rows:,}")
print("Row groups:", merged_pf.num_row_groups)
print("Missing Game_Genres rate:", f"{missing_rate:.2%}")
print(merged_pf.schema)

Merged row group 10/165 | rows so far: 7,269,899
Merged row group 20/165 | rows so far: 14,574,448
Merged row group 30/165 | rows so far: 21,451,147
Merged row group 40/165 | rows so far: 28,378,247
Merged row group 50/165 | rows so far: 34,967,771
Merged row group 60/165 | rows so far: 42,051,567
Merged row group 70/165 | rows so far: 48,770,700
Merged row group 80/165 | rows so far: 55,720,407
Merged row group 90/165 | rows so far: 62,499,854
Merged row group 100/165 | rows so far: 69,474,615
Merged row group 110/165 | rows so far: 76,946,194
Merged row group 120/165 | rows so far: 83,850,587
Merged row group 130/165 | rows so far: 90,483,411
Merged row group 140/165 | rows so far: 97,306,730
Merged row group 150/165 | rows so far: 104,565,207
Merged row group 160/165 | rows so far: 110,818,509
Merged row group 165/165 | rows so far: 113,883,717

Merged parquet complete.
Rows: 113,883,717
Row groups: 165
Missing Game_Genres rate: 4.78%
required group field_id=-1 schema {
  optional i